# PSTU DataThon 2026 Vol 1 — Improved Pipeline (CatBoost-centered)

**Goal:** beat the previous CatBoost Macro F1 of **0.692864** (AUC 0.896801) using a
CatBoost-centered hybrid ensemble with strict OOF validation.

**What changed vs the previous notebook:**
  - Removed the broken CNN (AUC=0.50, wastes GPU memory).
  - **Staged CatBoost hyperparameter search** (depth × lr × l2 × random_strength × bag_temp).
  - **Three CatBoost feature variants** (original / +row-stats / feature-selected).
  - **5-seed bagged** CatBoost + XGBoost (was 3 seeds).
  - HGB carried over.
  - Optional MLP included only if it improves OOF Macro F1.
  - Weighted ensemble search (step 0.02) over (CAT, XGB, HGB [, MLP]) with non-negativity and sum-to-1.
  - **Threshold is searched on OOF (0.05 - 0.50, step 0.005)** — never 0.5.
  - Optional pseudo-labeling with very high-confidence test rows as a single conservative experiment.
  - Final selection criterion: **highest OOF Macro F1**.

**Constraints respected:**
  - Only OOF predictions used for model selection, weight optimization, threshold tuning, and feature selection.
  - No test labels used anywhere. Submission uses the OOF-derived threshold.
  - GPU memory < 4 GB target. CatBoost/XGBoost fall back to CPU when GPU would be unsafe.
  - No CNN. MLP kept small and only added if it helps.

Sections:
  1. Imports & configuration
  2. Data loading
  3. EDA summary
  4. Preprocessing
  5. Feature engineering (row stats + missing + quantiles)
  6. Stratified OOF framework
  7. CatBoost hyperparameter search (staged)
  8. Best CatBoost seed/fold bagging (5 seeds)
  9. XGBoost seed/fold bagging (5 seeds)
 10. HGB
 11. Optional MLP
 12. OOF correlation
 13. Weighted ensemble search
 14. Threshold optimization
 15. Optional pseudo-labeling experiment
 16. Final model selection
 17. Test prediction
 18. Submission generation
 19. Final results table


## 1. Imports & Configuration


In [24]:
import os, gc, json, time, math, warnings, itertools, sys
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
pd.set_option('display.max_columns', 120)
pd.set_option('display.width', 200)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

import xgboost as xgb
from catboost import CatBoostClassifier

SEED = 42
N_SPLITS = 5
SEEDS_BAG = [42, 1024, 2025, 7, 99]   # 5-seed bagging for strongest models
EPOCHS_ANN = 30
BATCH_ANN = 1024
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

def set_seed(seed=SEED):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
set_seed(SEED)

print(f'Python: {sys.version.split()[0]} | Torch: {torch.__version__} | Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}  '
          f'({torch.cuda.get_device_properties(0).total_memory/1e9:.2f} GiB)')


Python: 3.10.7 | Torch: 2.5.1+cu121 | Device: cuda
GPU: NVIDIA GeForce RTX 2050  (4.29 GiB)


## 2. Data Loading


In [25]:
if os.path.exists('/kaggle/input'):
    DATA_DIR = '/kaggle/input/pstu-data-thon-2026-vol-1'
else:
    DATA_DIR = r'd:\DS\kaggle\PSTU_Datathon'

TRAIN_PATH = os.path.join(DATA_DIR, 'train.csv')
TEST_PATH  = os.path.join(DATA_DIR, 'test.csv')
SAMPLE_SUB_PATH = os.path.join(DATA_DIR, 'sample_submission.csv')
OUT_DIR = '/kaggle/working' if os.path.exists('/kaggle') else DATA_DIR

for p in [TRAIN_PATH, TEST_PATH, SAMPLE_SUB_PATH]:
    print(f'{p}  exists={os.path.exists(p)}')

train = pd.read_csv(TRAIN_PATH)
test  = pd.read_csv(TEST_PATH)
sample_sub = pd.read_csv(SAMPLE_SUB_PATH)

print('train:', train.shape, '  test:', test.shape, '  sample_sub:', sample_sub.shape)
print('id in train?', 'id' in train.columns, '| in test?', 'id' in test.columns)

# `id` only exists in test — preserve sample_sub ordering for the final submission.
test_ids = test['id'].values
y = train['TARGET'].values.astype(np.int64)
print('pos rate:', y.mean())


d:\DS\kaggle\PSTU_Datathon\train.csv  exists=True
d:\DS\kaggle\PSTU_Datathon\test.csv  exists=True
d:\DS\kaggle\PSTU_Datathon\sample_submission.csv  exists=True
train: (76020, 351)   test: (60654, 351)   sample_sub: (60654, 2)
id in train? False | in test? True
pos rate: 0.0395685345961589


## 3. EDA Summary


In [26]:
feat_cols = [c for c in train.columns if c.startswith('feat_')]
print(f'#feature columns: {len(feat_cols)}')

print('\nTarget distribution:'); print(pd.Series(y).value_counts().sort_index())
print(f'Class balance: 0 = {(y==0).mean()*100:.2f}%, 1 = {(y==1).mean()*100:.2f}%')

obj_cols = [c for c in feat_cols if train[c].dtype == object]
print(f'\n#object-typed feature columns: {len(obj_cols)} -> {obj_cols}')

nunique = train[feat_cols].nunique()
const_cols = nunique[nunique <= 1].index.tolist()
print(f'#constant (nunique<=1) columns: {len(const_cols)}')

miss = train[feat_cols].isna().mean()
print(f'NaN ratio: mean={miss.mean():.4f}, max={miss.max():.4f}')
print(f'#all-NaN cols: {(miss==1).sum()} | #>=20% NaN: {(miss>=0.2).sum()}')

print('\nDtype counts:'); print(train[feat_cols].dtypes.value_counts())


#feature columns: 350

Target distribution:
0    73012
1     3008
Name: count, dtype: int64
Class balance: 0 = 96.04%, 1 = 3.96%

#object-typed feature columns: 6 -> ['feat_142', 'feat_157', 'feat_318', 'feat_320', 'feat_325', 'feat_337']
#constant (nunique<=1) columns: 28
NaN ratio: mean=0.0000, max=0.0000
#all-NaN cols: 0 | #>=20% NaN: 0

Dtype counts:
int64      212
float64    132
object       6
Name: count, dtype: int64


## 4. Preprocessing


Steps:
  1. Drop **constant columns** (nunique ≤ 1).
  2. Label-encode the 6 object-typed feature columns (combined train+test vocabulary to be safe).
  3. Outlier-clip numeric features to the 0.2 / 99.8 percentile (asymmetric to keep the heavy tails on the positive side).
  4. Build **CatBoost DataFrame** (cat cols as int32, numeric as float32) and a separate **imputed+clipped float32 matrix** for downstream models.

All statistics (medians, clip bounds) are computed from **train only** to avoid leakage.


In [27]:
# 4.1 Drop constant columns
GLOBAL_CONST = [c for c in feat_cols if train[c].nunique() <= 1]
KEEP_COLS = [c for c in feat_cols if c not in GLOBAL_CONST]
print(f'Kept {len(KEEP_COLS)} features after dropping {len(GLOBAL_CONST)} constant columns.')

# 4.2 Label-encode object columns (train+test combined vocab)
cat_maps = {}
for c in obj_cols:
    combined = pd.concat([train[c].astype(str), test[c].astype(str)], axis=0).fillna('__NAN__')
    le = LabelEncoder().fit(combined)
    train[c] = le.transform(train[c].astype(str).fillna('__NAN__'))
    test[c]  = le.transform(test[c].astype(str).fillna('__NAN__'))
    cat_maps[c] = le
print('After encoding, object-typed feature columns:',
      sum(train[c].dtype == object for c in train.columns if c.startswith('feat_')))

# 4.3 Outlier clipping (train statistics)
LO_Q, HI_Q = 0.002, 0.998
NUM_COLS = [c for c in KEEP_COLS if c not in obj_cols]
lo = train[NUM_COLS].quantile(LO_Q)
hi = train[NUM_COLS].quantile(HI_Q)
for c in NUM_COLS:
    train[c] = train[c].clip(lo[c], hi[c])
    test[c]  = test[c].clip(lo[c], hi[c])
print(f'Clipped numeric features to [{LO_Q}, {HI_Q}] percentiles (train-derived).')

# 4.4 Median-impute (defensive — train has 0 NaN but be safe) and build float32 matrix
med = train[KEEP_COLS].median()
X_gbdt = train[KEEP_COLS].fillna(med).astype(np.float32).values
X_test_gbdt = test[KEEP_COLS].fillna(med).astype(np.float32).values
print(f'X_gbdt: {X_gbdt.shape} | X_test_gbdt: {X_test_gbdt.shape} | y: {y.shape}')
print(f'NaNs remaining -> train: {np.isnan(X_gbdt).sum()}, test: {np.isnan(X_test_gbdt).sum()}')
print(f'Class balance: {np.bincount(y)}')

# 4.5 CatBoost DataFrame view (cat as int32, numeric as float32 — CatBoost refuses cat_features on float arrays)
X_cat_df = pd.DataFrame(index=train.index)
X_test_cat_df = pd.DataFrame(index=test.index)
for c in KEEP_COLS:
    if c in obj_cols:
        X_cat_df[c] = train[c].astype('int32')
        X_test_cat_df[c] = test[c].astype('int32')
    else:
        X_cat_df[c] = train[c].astype('float32')
        X_test_cat_df[c] = test[c].astype('float32')
print(f'X_cat_df: {X_cat_df.shape}  (cat-cols: {len(obj_cols)})')

# Category indices (for CatBoost)
CAT_IDX = [i for i, c in enumerate(KEEP_COLS) if c in obj_cols]
print(f'CatBoost cat indices: {CAT_IDX} -> {[KEEP_COLS[i] for i in CAT_IDX]}')


Kept 322 features after dropping 28 constant columns.
After encoding, object-typed feature columns: 0
Clipped numeric features to [0.002, 0.998] percentiles (train-derived).
X_gbdt: (76020, 322) | X_test_gbdt: (60654, 322) | y: (76020,)
NaNs remaining -> train: 0, test: 0
Class balance: [73012  3008]
X_cat_df: (76020, 322)  (cat-cols: 6)
CatBoost cat indices: [130, 145, 291, 292, 296, 308] -> ['feat_142', 'feat_157', 'feat_318', 'feat_320', 'feat_325', 'feat_337']


## 5. Feature Engineering (row-level statistics)


For the **feature variant B** (CatBoost + row stats), we add:
  - row mean / std / min / max / median / sum / range
  - missing-value count (across the 350 features)
  - zero-value count (across numeric features)
  - negative-value count (across numeric features)
  - selected quantiles (Q1, Q25, Q75, Q99 — train-derived reference only)

These statistics are computed **after** the outlier clipping in step 4, on the train-derived `KEEP_COLS`.


In [28]:
def row_stats(df: pd.DataFrame, cols=KEEP_COLS, num_cols=NUM_COLS, ref_quantiles=None):
    """Return a DataFrame of row-level statistics. If ref_quantiles is given,
    reuse those quantile values (computed on train) to compute the same columns for test."""
    out = pd.DataFrame(index=df.index)
    v = df[cols].astype('float32')
    out['row_mean']   = v.mean(axis=1)
    out['row_std']    = v.std(axis=1)
    out['row_min']    = v.min(axis=1)
    out['row_max']    = v.max(axis=1)
    out['row_median'] = v.median(axis=1)
    out['row_sum']    = v.sum(axis=1)
    out['row_range']  = out['row_max'] - out['row_min']
    out['row_nan']    = df[cols].isna().sum(axis=1).astype('float32')
    out['row_zero']   = (df[num_cols] == 0).sum(axis=1).astype('float32')
    out['row_neg']    = (df[num_cols] < 0).sum(axis=1).astype('float32')
    if ref_quantiles is None:
        ref = v.quantile([0.01, 0.25, 0.75, 0.99])
    else:
        ref = ref_quantiles
    for q in [0.01, 0.25, 0.75, 0.99]:
        out[f'row_q{int(q*100)}'] = (v < ref.loc[q]).sum(axis=1).astype('float32')  # per-row count of features below this train-quantile
    return out, ref

row_train, ROW_REF = row_stats(train)
row_test,  _       = row_stats(test, ref_quantiles=ROW_REF)
ROW_COLS = list(row_train.columns)
print(f'Row-stat features: {len(ROW_COLS)} -> {ROW_COLS}')

# Build CatBoost view B (original + row stats)
X_catB_df = pd.concat([X_cat_df, row_train.astype('float32')], axis=1)
X_test_catB_df = pd.concat([X_test_cat_df, row_test.astype('float32')], axis=1)
print(f'X_catB_df: {X_catB_df.shape}  | X_test_catB_df: {X_test_catB_df.shape}')

# Numeric (gbdt) view including row stats — for XGB / HGB / MLP
X_gbdtB = np.concatenate([X_gbdt, row_train.values.astype(np.float32)], axis=1)
X_test_gbdtB = np.concatenate([X_test_gbdt, row_test.values.astype(np.float32)], axis=1)
print(f'X_gbdtB: {X_gbdtB.shape}  | X_test_gbdtB: {X_test_gbdtB.shape}')

ROW_COLS_NUM = list(range(X_gbdt.shape[1], X_gbdt.shape[1] + len(ROW_COLS)))
gc.collect()


Row-stat features: 14 -> ['row_mean', 'row_std', 'row_min', 'row_max', 'row_median', 'row_sum', 'row_range', 'row_nan', 'row_zero', 'row_neg', 'row_q1', 'row_q25', 'row_q75', 'row_q99']
X_catB_df: (76020, 336)  | X_test_catB_df: (60654, 336)
X_gbdtB: (76020, 336)  | X_test_gbdtB: (60654, 336)


12620

## 6. Stratified OOF Framework


In [29]:
SKF = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
FOLDS = list(SKF.split(np.zeros(len(y)), y))
print(f'Stratified {N_SPLITS}-fold splits: train sizes {[len(tr) for tr,_ in FOLDS]}')

def best_threshold(y_true, y_prob, lo=0.05, hi=0.50, step=0.005):
    """Find the threshold that maximizes Macro F1 on the given OOF probability."""
    grid = np.arange(lo, hi + step/2, step)
    best_t, best_f1 = 0.5, -1.0
    for t in grid:
        f1 = f1_score(y_true, (y_prob >= t).astype(int), average='macro')
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

def report_oof(y_true, oof_prob, name='model'):
    auc = roc_auc_score(y_true, oof_prob)
    t, f1 = best_threshold(y_true, oof_prob)
    pos_pct = (oof_prob >= t).mean() * 100.0
    print(f'  {name:38s} | AUC={auc:.5f} | MacroF1={f1:.5f} | t={t:.3f} | pos%={pos_pct:.2f}')
    return t, f1, auc

def cat_task_type(use_gpu=True):
    """Return 'GPU' if safe (>=4 GiB), else 'CPU'. CatBoost on GPU needs the train pool on device;"""
    """On 4 GiB GPUs we fall back to CPU to avoid OOM during pooling."""
    if not use_gpu or not torch.cuda.is_available():
        return 'CPU'
    try:
        free_mem = torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_reserved(0)
        if free_mem / 1e9 < 1.0:
            return 'CPU'
    except Exception:
        return 'CPU'
    return 'GPU'

print(f'CatBoost task_type = {cat_task_type()}')


Stratified 5-fold splits: train sizes [60816, 60816, 60816, 60816, 60816]
CatBoost task_type = GPU


## 7. CatBoost Hyperparameter Search (staged)


Staged search to avoid an explosive grid (4 × 3 × 5 × 4 × 4 = 960 configs):
  - **Stage 1:** fix depth and lr, sweep `l2_leaf_reg` on each (depth, lr) — keep top-3 per (depth, lr).
  - **Stage 2:** fix the best (depth, lr, l2), sweep `random_strength` and `bagging_temperature`.
  - **Stage 3:** keep the top 3 configs from the union, finalize.

Each config uses **early stopping** (200 rounds) on the validation fold, **single seed** at the search
stage (to keep time bounded), feature variant A (original).

Selection criterion: **OOF Macro F1** (AUC used as a secondary check).


In [31]:
def train_cat_one_fold(Xtr_df, ytr, Xva_df, yva, params, cat_idx):
    m = CatBoostClassifier(loss_function='Logloss', eval_metric='AUC',
                           iterations=4000, verbose=False, early_stopping_rounds=200, **params)
    m.fit(Xtr_df, ytr, eval_set=(Xva_df, yva), cat_features=cat_idx, use_best_model=True, verbose=False)
    pv = m.predict_proba(Xva_df)[:, 1]
    return pv, m.get_best_iteration()

def eval_cat_config(params, X_df, cat_idx, seed=SEED):
    """Train CatBoost with the given params across all folds and return OOF probability.
    Test predictions are NOT computed during the search (saves ~30% wall time); only OOF is needed
    to pick the best hyperparameters. Final retraining in section 8 uses 5-seed bagging and produces
    test predictions."""
    oof = np.zeros(len(X_df), dtype=np.float32)
    for fold, (tr, va) in enumerate(FOLDS):
        Xtr_df = X_df.iloc[tr].reset_index(drop=True)
        Xva_df = X_df.iloc[va].reset_index(drop=True)
        pv, _ = train_cat_one_fold(Xtr_df, y[tr], Xva_df, y[va], params, cat_idx)
        oof[va] = pv
    auc = roc_auc_score(y, oof)
    t, f1 = best_threshold(y, oof)
    return oof, auc, t, f1

# Fix params builder
def cat_params(depth, lr, l2, rs, bt, seed=SEED, task_type=None):
    # CatBoost constraint: bagging_temperature is only valid with Bayesian bootstrap.
    # For Bernoulli bootstrap, use subsample (sampling fraction) instead.
    if task_type is None:
        task_type = cat_task_type()
    if bt > 0:
        boot = 'Bayesian'
        boot_kwargs = dict(bagging_temperature=bt)
    else:
        boot = 'Bernoulli'
        # Map bt in [0,2] -> subsample in [1.0, 0.5] so a value of 0 means 'no row
        # subsampling' and higher bt means more aggressive subsampling.
        boot_kwargs = dict(subsample=max(0.5, 1.0 - 0.25 * bt))
    return dict(
        learning_rate=lr, depth=depth, l2_leaf_reg=l2,
        random_strength=rs,
        random_seed=seed, task_type=task_type,
        bootstrap_type=boot, **boot_kwargs,
    )

search_results = []
t0 = time.time()
print(f'[{time.strftime("%H:%M:%S")}] Starting CatBoost staged search...')

# Stage 1: depth in {6,7,8}, lr in {0.02, 0.03, 0.05}, l2 in {3, 5, 8} -> 27 configs
stage1_results = []
for depth in [8]:
    for lr in [0.02, 0.03, 0.05]:
        for l2 in [ 5, 8]:
# for depth in [6, 7, 8]:
#     for lr in [0.02, 0.03, 0.05]:
#         for l2 in [3, 5, 8]:
            params = cat_params(depth, lr, l2, rs=1.0, bt=1.0)
            oof, auc, t, f1 = eval_cat_config(params, X_cat_df, CAT_IDX)
            stage1_results.append((depth, lr, l2, 1.0, 1.0, auc, t, f1))
            print(f'  d={depth} lr={lr} l2={l2:>2} | AUC={auc:.5f} | F1={f1:.5f} t={t:.3f}')
stage1_results.sort(key=lambda r: r[7], reverse=True)  # sort by Macro F1
print(f'\nStage 1 done in {time.time()-t0:.1f}s. Top 5 by Macro F1:')
for r in stage1_results[:5]:
    print(f'  d={r[0]} lr={r[1]} l2={r[2]} rs={r[3]} bt={r[4]} | AUC={r[5]:.5f} F1={r[6]:.3f} (t={r[7]:.3f})')
    print(f'  (showing F1 then AUC)'); break


[01:44:00] Starting CatBoost staged search...


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


  d=8 lr=0.02 l2= 5 | AUC=0.89576 | F1=0.69114 t=0.190


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


  d=8 lr=0.02 l2= 8 | AUC=0.89654 | F1=0.69446 t=0.190


...

  d=8 lr=0.03 l2= 8 | AUC=0.89558 | F1=0.69186 t=0.175


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


  d=8 lr=0.05 l2= 5 | AUC=0.89500 | F1=0.68847 t=0.175


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


  d=8 lr=0.05 l2= 8 | AUC=0.89446 | F1=0.69082 t=0.180

Stage 1 done in 3506.6s. Top 5 by Macro F1:
  d=8 lr=0.02 l2=8 rs=1.0 bt=1.0 | AUC=0.89654 F1=0.190 (t=0.694)
  (showing F1 then AUC)


In [32]:
# Stage 2: take top-3 from stage 1 and sweep (rs, bt) on their (depth, lr, l2) -> 3*16=48 configs
stage2_results = list(stage1_results[:3])
for base in stage1_results[:3]:
    d, lr, l2, _, _, _, _, _ = base
    for rs in [0, 0.5, 1, 2]:
        for bt in [0, 0.5, 1, 2]:
            if rs == 1.0 and bt == 1.0 and (d, lr, l2, 1.0, 1.0) in [(r[0],r[1],r[2],r[3],r[4]) for r in stage2_results]:
                continue  # already computed during stage 1
            params = cat_params(d, lr, l2, rs=rs, bt=bt)
            oof, auc, t, f1 = eval_cat_config(params, X_cat_df, CAT_IDX)
            stage2_results.append((d, lr, l2, rs, bt, auc, t, f1))
            print(f'  d={d} lr={lr} l2={l2} rs={rs} bt={bt} | AUC={auc:.5f} F1={f1:.5f} t={t:.3f}')

stage2_results.sort(key=lambda r: r[7], reverse=True)
print(f'\nStage 2 done. Top 5 by Macro F1:')
for r in stage2_results[:5]:
    print(f'  d={r[0]} lr={r[1]} l2={r[2]} rs={r[3]} bt={r[4]} | AUC={r[5]:.5f} F1={r[6]:.3f} (t={r[7]:.3f})')

BEST_CAT_PARAMS = cat_params(stage2_results[0][0], stage2_results[0][1], stage2_results[0][2],
                              stage2_results[0][3], stage2_results[0][4])
print(f'\nBest CatBoost params: {BEST_CAT_PARAMS}')
gc.collect()


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


  d=8 lr=0.02 l2=8 rs=0 bt=0 | AUC=0.89464 F1=0.68956 t=0.195


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


  d=8 lr=0.02 l2=8 rs=0 bt=0.5 | AUC=0.89489 F1=0.68911 t=0.195


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


...

Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


  d=8 lr=0.02 l2=5 rs=2 bt=1 | AUC=0.89573 F1=0.69062 t=0.180


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU


  d=8 lr=0.02 l2=5 rs=2 bt=2 | AUC=0.89459 F1=0.69024 t=0.190

Stage 2 done. Top 5 by Macro F1:
  d=8 lr=0.02 l2=8 rs=1.0 bt=1.0 | AUC=0.89654 F1=0.190 (t=0.694)
  d=8 lr=0.03 l2=8 rs=0.5 bt=1 | AUC=0.89469 F1=0.195 (t=0.692)
  d=8 lr=0.03 l2=8 rs=2 bt=0 | AUC=0.89580 F1=0.180 (t=0.692)
  d=8 lr=0.02 l2=5 rs=0.5 bt=1 | AUC=0.89514 F1=0.195 (t=0.692)
  d=8 lr=0.03 l2=8 rs=1.0 bt=1.0 | AUC=0.89558 F1=0.175 (t=0.692)

Best CatBoost params: {'learning_rate': 0.02, 'depth': 8, 'l2_leaf_reg': 8, 'random_strength': 1.0, 'random_seed': 42, 'task_type': 'GPU', 'bootstrap_type': 'Bayesian', 'bagging_temperature': 1.0}


127

## 8. Best CatBoost — 5-seed bagging


Best CatBoost hyperparameters from section 7, retrained with **5 seeds** across the 5 folds.
Test predictions are averaged across all `(seed, fold)` runs; OOF is averaged across seeds for each fold.

We also compute the configuration **B (with row stats)** and the **feature-selected variant** here so we can compare.


In [33]:
def train_cat_bagged(X_df, X_test_df, cat_idx, params, seeds=SEEDS_BAG, name='cat'):
    oof = np.zeros(len(X_df), dtype=np.float32)
    test_pred = np.zeros(len(X_test_df), dtype=np.float32)
    for seed in seeds:
        params_s = dict(params); params_s['random_seed'] = seed
        for fold, (tr, va) in enumerate(FOLDS):
            Xtr_df = X_df.iloc[tr].reset_index(drop=True)
            Xva_df = X_df.iloc[va].reset_index(drop=True)
            Xte_df = X_test_df.reset_index(drop=True)
            m = CatBoostClassifier(loss_function='Logloss', eval_metric='AUC',
                                   iterations=4000, verbose=False,
                                   early_stopping_rounds=200, **params_s)
            m.fit(Xtr_df, y[tr], eval_set=(Xva_df, y[va]), cat_features=cat_idx,
                  use_best_model=True, verbose=False)
            pv = m.predict_proba(Xva_df)[:, 1]
            pt = m.predict_proba(Xte_df)[:, 1]
            oof[va] += pv / len(seeds)
            test_pred += pt / (N_SPLITS * len(seeds))
            print(f'  {name} seed={seed:>4} fold={fold+1}  best_iter={m.get_best_iteration()}')
    return oof, test_pred

print('=== CatBoost (original features) — 5-seed bag ===')
oof_catA, test_catA = train_cat_bagged(X_cat_df, X_test_cat_df, CAT_IDX, BEST_CAT_PARAMS, name='catA')
t_catA, f1_catA, auc_catA = report_oof(y, oof_catA, 'CatBoost-A (original, 5-seed)')
gc.collect()


=== CatBoost (original features) — 5-seed bag ===


Default metric period is 5 because AUC is/are not implemented for GPU


  catA seed=  42 fold=1  best_iter=1164


Default metric period is 5 because AUC is/are not implemented for GPU


  catA seed=  42 fold=2  best_iter=904


...

Default metric period is 5 because AUC is/are not implemented for GPU


  catA seed=  99 fold=4  best_iter=966


Default metric period is 5 because AUC is/are not implemented for GPU


  catA seed=  99 fold=5  best_iter=1628
  CatBoost-A (original, 5-seed)          | AUC=0.89685 | MacroF1=0.69206 | t=0.205 | pos%=3.78


0

In [34]:
print('=== CatBoost (original + row stats) — 5-seed bag ===')
oof_catB, test_catB = train_cat_bagged(X_catB_df, X_test_catB_df, CAT_IDX, BEST_CAT_PARAMS, name='catB')
t_catB, f1_catB, auc_catB = report_oof(y, oof_catB, 'CatBoost-B (+row stats, 5-seed)')
gc.collect()


=== CatBoost (original + row stats) — 5-seed bag ===


Default metric period is 5 because AUC is/are not implemented for GPU


  catB seed=  42 fold=1  best_iter=1481


Default metric period is 5 because AUC is/are not implemented for GPU


  catB seed=  42 fold=2  best_iter=1668


...

Default metric period is 5 because AUC is/are not implemented for GPU


  catB seed=  99 fold=4  best_iter=1283


Default metric period is 5 because AUC is/are not implemented for GPU


  catB seed=  99 fold=5  best_iter=1589
  CatBoost-B (+row stats, 5-seed)        | AUC=0.89663 | MacroF1=0.69237 | t=0.185 | pos%=4.31


0

## 9. XGBoost — 5-seed bagging


In [35]:
USE_GPU_XGB = False  # 4 GiB GPU is too tight for XGB GPU; CPU is fine on this dataset size
xgb_task = 'hist'
def train_xgb_bagged(X_tr, X_te, seeds=SEEDS_BAG, name='xgb'):
    oof = np.zeros(len(X_tr), dtype=np.float32)
    test_pred = np.zeros(len(X_te), dtype=np.float32)
    for seed in seeds:
        for fold, (tr, va) in enumerate(FOLDS):
            params = dict(
                objective='binary:logistic', eval_metric=['auc', 'logloss'],
                tree_method='hist', device='cuda' if USE_GPU_XGB else 'cpu',
                learning_rate=0.03, max_depth=6, min_child_weight=4,
                subsample=0.8, colsample_bytree=0.7, reg_alpha=1.0, reg_lambda=2.0, gamma=0.0,
                seed=seed, n_jobs=os.cpu_count(),
            )
            dtr = xgb.DMatrix(X_tr[tr], label=y[tr])
            dva = xgb.DMatrix(X_tr[va], label=y[va])
            dte = xgb.DMatrix(X_te)
            booster = xgb.train(params, dtr, num_boost_round=4000,
                                evals=[(dva, 'va')], early_stopping_rounds=200, verbose_eval=False)
            pv = booster.predict(dva, iteration_range=(0, booster.best_iteration + 1))
            pt = booster.predict(dte, iteration_range=(0, booster.best_iteration + 1))
            oof[va] += pv / len(seeds)
            test_pred += pt / (N_SPLITS * len(seeds))
            print(f'  {name} seed={seed:>4} fold={fold+1}  best_iter={booster.best_iteration}')
    return oof, test_pred

print('=== XGBoost (with row stats) — 5-seed bag ===')
oof_xgb, test_xgb = train_xgb_bagged(X_gbdtB, X_test_gbdtB, name='xgb')
t_xgb, f1_xgb, auc_xgb = report_oof(y, oof_xgb, 'XGBoost (5-seed, +row stats)')
gc.collect()


=== XGBoost (with row stats) — 5-seed bag ===
  xgb seed=  42 fold=1  best_iter=305
  xgb seed=  42 fold=2  best_iter=296
  xgb seed=  42 fold=3  best_iter=325
  xgb seed=  42 fold=4  best_iter=262
  xgb seed=  42 fold=5  best_iter=333
  xgb seed=1024 fold=1  best_iter=341
  xgb seed=1024 fold=2  best_iter=263
  xgb seed=1024 fold=3  best_iter=328
  xgb seed=1024 fold=4  best_iter=327
  xgb seed=1024 fold=5  best_iter=402
  xgb seed=2025 fold=1  best_iter=311
  xgb seed=2025 fold=2  best_iter=348
  xgb seed=2025 fold=3  best_iter=417
  xgb seed=2025 fold=4  best_iter=252
  xgb seed=2025 fold=5  best_iter=328
  xgb seed=   7 fold=1  best_iter=364
  xgb seed=   7 fold=2  best_iter=351
  xgb seed=   7 fold=3  best_iter=373
  xgb seed=   7 fold=4  best_iter=322
  xgb seed=   7 fold=5  best_iter=357
  xgb seed=  99 fold=1  best_iter=301
  xgb seed=  99 fold=2  best_iter=313
  xgb seed=  99 fold=3  best_iter=324
  xgb seed=  99 fold=4  best_iter=333
  xgb seed=  99 fold=5  best_iter=407
  XG

2

## 10. HGB


In [36]:
print('=== HistGradientBoosting (with row stats) ===')
oof_hgb = np.zeros(len(X_gbdtB), dtype=np.float32)
test_hgb = np.zeros(len(X_test_gbdtB), dtype=np.float32)
for fold, (tr, va) in enumerate(FOLDS):
    m = HistGradientBoostingClassifier(
        learning_rate=0.05, max_iter=2000, max_leaf_nodes=63, min_samples_leaf=64,
        l2_regularization=1.0, early_stopping=True, validation_fraction=0.15, n_iter_no_change=50,
        random_state=SEED,
    )
    m.fit(X_gbdtB[tr], y[tr])
    pv = m.predict_proba(X_gbdtB[va])[:, 1]
    pt = m.predict_proba(X_test_gbdtB)[:, 1]
    oof_hgb[va] = pv
    test_hgb += pt / N_SPLITS
    print(f'  HGB fold={fold+1}  iters={m.n_iter_}')
t_hgb, f1_hgb, auc_hgb = report_oof(y, oof_hgb, 'HGB (+row stats)')
gc.collect()


=== HistGradientBoosting (with row stats) ===
  HGB fold=1  iters=171
  HGB fold=2  iters=153
  HGB fold=3  iters=136
  HGB fold=4  iters=146
  HGB fold=5  iters=178
  HGB (+row stats)                       | AUC=0.88396 | MacroF1=0.67350 | t=0.175 | pos%=4.51


0

## 11. Optional MLP (only included if it helps ensemble)


In [37]:
# Quantile transform + standardize (fit on full train, since this is a 'view' transform)
from sklearn.preprocessing import QuantileTransformer
QT = QuantileTransformer(output_distribution='normal', n_quantiles=2000, subsample=200000, random_state=SEED)
X_nn_full = QT.fit_transform(X_gbdtB).astype(np.float32)
X_nn_test_full = QT.transform(X_test_gbdtB).astype(np.float32)
SC = StandardScaler()
X_nn_full = SC.fit_transform(X_nn_full).astype(np.float32)
X_nn_test_full = SC.transform(X_nn_test_full).astype(np.float32)
print(f'X_nn_full: {X_nn_full.shape} | X_nn_test_full: {X_nn_test_full.shape}')

class MLP(nn.Module):
    def __init__(self, in_dim, hidden=(256, 128), dropout=0.3):
        super().__init__()
        layers = []
        d = in_dim
        for h in hidden:
            layers += [nn.Linear(d, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        layers += [nn.Linear(d, 1)]
        self.net = nn.Sequential(*layers)
    def forward(self, x): return self.net(x).squeeze(-1)

def train_mlp_fold(Xtr, ytr, Xva, yva, Xte, seed=SEED, epochs=EPOCHS_ANN, batch_size=BATCH_ANN,
                    lr=1e-3, wd=1e-4, dropout=0.3, patience=6, verbose=False):
    set_seed(seed + 13)
    device = DEVICE
    model = MLP(Xtr.shape[1], hidden=(256, 128), dropout=dropout).to(device)
    pos_w = torch.tensor([float((ytr == 0).sum() / max(1, (ytr == 1).sum()))], device=device)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=pos_w)
    optim = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)
    Xtr_t = torch.from_numpy(Xtr); ytr_t = torch.from_numpy(ytr.astype(np.float32))
    Xva_t = torch.from_numpy(Xva).to(device); yva_t = torch.from_numpy(yva.astype(np.float32)).to(device)
    Xte_t = torch.from_numpy(Xte).to(device)
    loader = DataLoader(TensorDataset(Xtr_t, ytr_t), batch_size=batch_size, shuffle=True, drop_last=False)
    best_auc, best_state, bad = -1.0, None, 0
    for epoch in range(epochs):
        model.train()
        for xb, yb in loader:
            xb = xb.to(device, non_blocking=True); yb = yb.to(device, non_blocking=True)
            optim.zero_grad()
            loss = loss_fn(model(xb), yb)
            loss.backward(); optim.step()
        sched.step()
        model.eval()
        with torch.no_grad():
            va_prob = torch.sigmoid(model(Xva_t)).cpu().numpy()
        va_auc = roc_auc_score(yva, va_prob)
        if verbose: print(f'    epoch {epoch+1:02d}  val AUC={va_auc:.5f}')
        if va_auc > best_auc + 1e-5:
            best_auc, best_state, bad = va_auc, {k: v.detach().clone() for k, v in model.state_dict().items()}, 0
        else:
            bad += 1
            if bad >= patience: break
    model.load_state_dict(best_state); model.eval()
    with torch.no_grad():
        va_prob = torch.sigmoid(model(Xva_t)).cpu().numpy()
        te_prob = torch.sigmoid(model(Xte_t)).cpu().numpy()
    return va_prob, te_prob, best_auc

print('=== MLP (single seed, 5-fold) ===')
oof_mlp = np.zeros(len(X_nn_full), dtype=np.float32)
test_mlp = np.zeros(len(X_nn_test_full), dtype=np.float32)
for fold, (tr, va) in enumerate(FOLDS):
    pv, pt, va_auc = train_mlp_fold(X_nn_full[tr], y[tr], X_nn_full[va], y[va], X_nn_test_full, seed=SEED)
    oof_mlp[va] = pv
    test_mlp += pt / N_SPLITS
    print(f'  MLP fold={fold+1}  AUC={va_auc:.5f}')
t_mlp, f1_mlp, auc_mlp = report_oof(y, oof_mlp, 'MLP')
del X_nn_full, X_nn_test_full, QT, SC
gc.collect()
if torch.cuda.is_available(): torch.cuda.empty_cache()


X_nn_full: (76020, 336) | X_nn_test_full: (60654, 336)
=== MLP (single seed, 5-fold) ===
  MLP fold=1  AUC=0.86428
  MLP fold=2  AUC=0.87487
  MLP fold=3  AUC=0.87808
  MLP fold=4  AUC=0.87728
  MLP fold=5  AUC=0.88049
  MLP                                    | AUC=0.87461 | MacroF1=0.55439 | t=0.500 | pos%=23.63


## 12. OOF Prediction Correlation


In [38]:
oof_df = pd.DataFrame({
    'catA': oof_catA, 'catB': oof_catB, 'xgb': oof_xgb, 'hgb': oof_hgb, 'mlp': oof_mlp,
})
print('Pearson correlation between OOF predictions:'); print(oof_df.corr().round(3))
print('\nSpearman correlation between OOF predictions:'); print(oof_df.corr(method='spearman').round(3))


Pearson correlation between OOF predictions:
       catA   catB    xgb    hgb    mlp
catA  1.000  0.997  0.970  0.926  0.646
catB  0.997  1.000  0.970  0.927  0.645
xgb   0.970  0.970  1.000  0.953  0.652
hgb   0.926  0.927  0.953  1.000  0.607
mlp   0.646  0.645  0.652  0.607  1.000

Spearman correlation between OOF predictions:
       catA   catB    xgb    hgb    mlp
catA  1.000  0.998  0.962  0.925  0.842
catB  0.998  1.000  0.962  0.925  0.840
xgb   0.962  0.962  1.000  0.958  0.859
hgb   0.925  0.925  0.958  1.000  0.832
mlp   0.842  0.840  0.859  0.832  1.000


## 13. Weighted Ensemble Search


Search convex weights on a 0.02 grid:
  - 3-way (CAT, XGB, HGB): every (w_cat, w_xgb, w_hgb) with each in {0.0, 0.02, 0.04, ..., 1.0} and sum=1.
  - 4-way (CAT, XGB, HGB, MLP): same idea, step 0.05 (4-way grid is too dense at 0.02).
  - Dropouts (drop one model, equal weights on the rest) are also tried.
  - Single-model baselines are included.

For each weight vector we compute OOF AUC and the best Macro F1 threshold on the OOF blend.
Selections favor **OOF Macro F1**.


In [39]:
def enumerate_simplex(n, step=0.02):
    """Yield non-negative weights (n) on a regular grid summing to 1."""
    grid = np.arange(0.0, 1.0 + step/2, step)
    if n == 1:
        yield (1.0,); return
    for w in itertools.product(grid, repeat=n):
        if abs(sum(w) - 1.0) < step/2:
            yield w

def evaluate_blend(weights, oof_mat, test_mat=None):
    blend = sum(w * oof_mat[i] for i, w in enumerate(weights))
    auc = roc_auc_score(y, blend)
    t, f1 = best_threshold(y, blend)
    return blend, auc, t, f1

# 3-way grid (CAT, XGB, HGB) using the better of catA/catB per call — we pick catA vs catB at the end.
print('=== 3-way (catA, xgb, hgb) simplex search (step 0.02) ===')
blend_results = []
for w_cat, w_xgb, w_hgb in enumerate_simplex(3, step=0.02):
    weights = (w_cat, w_xgb, w_hgb)
    oof_mat = [oof_catA, oof_xgb, oof_hgb]
    blend, auc, t, f1 = evaluate_blend(weights, oof_mat)
    blend_results.append(('catA,xgb,hgb', weights, auc, t, f1))
blend_results.sort(key=lambda r: r[4], reverse=True)
print('Top 5 3-way blends:');
for r in blend_results[:5]: print(f'  {r[0]} w={r[1]} | AUC={r[2]:.5f} F1={r[3]:.3f} (t={r[4]:.3f})')

print('\n=== 3-way (catB, xgb, hgb) simplex search (step 0.02) ===')
blend_results_B = []
for w_cat, w_xgb, w_hgb in enumerate_simplex(3, step=0.02):
    weights = (w_cat, w_xgb, w_hgb)
    oof_mat = [oof_catB, oof_xgb, oof_hgb]
    blend, auc, t, f1 = evaluate_blend(weights, oof_mat)
    blend_results_B.append(('catB,xgb,hgb', weights, auc, t, f1))
blend_results_B.sort(key=lambda r: r[4], reverse=True)
print('Top 5 3-way blends (catB):');
for r in blend_results_B[:5]: print(f'  {r[0]} w={r[1]} | AUC={r[2]:.5f} F1={r[3]:.3f} (t={r[4]:.3f})')

print('\n=== 4-way (catA, xgb, hgb, mlp) simplex search (step 0.05) ===')
blend_results_4 = []
for w in enumerate_simplex(4, step=0.05):
    w_cat, w_xgb, w_hgb, w_mlp = w
    oof_mat = [oof_catA, oof_xgb, oof_hgb, oof_mlp]
    blend, auc, t, f1 = evaluate_blend(w, oof_mat)
    blend_results_4.append(('catA,xgb,hgb,mlp', w, auc, t, f1))
blend_results_4.sort(key=lambda r: r[4], reverse=True)
print('Top 5 4-way blends:');
for r in blend_results_4[:5]: print(f'  {r[0]} w={r[1]} | AUC={r[2]:.5f} F1={r[3]:.3f} (t={r[4]:.3f})')

print('\n=== 4-way (catB, xgb, hgb, mlp) simplex search (step 0.05) ===')
blend_results_4B = []
for w in enumerate_simplex(4, step=0.05):
    w_cat, w_xgb, w_hgb, w_mlp = w
    oof_mat = [oof_catB, oof_xgb, oof_hgb, oof_mlp]
    blend, auc, t, f1 = evaluate_blend(w, oof_mat)
    blend_results_4B.append(('catB,xgb,hgb,mlp', w, auc, t, f1))
blend_results_4B.sort(key=lambda r: r[4], reverse=True)
print('Top 5 4-way blends (catB):');
for r in blend_results_4B[:5]: print(f'  {r[0]} w={r[1]} | AUC={r[2]:.5f} F1={r[3]:.3f} (t={r[4]:.3f})')


=== 3-way (catA, xgb, hgb) simplex search (step 0.02) ===
Top 5 3-way blends:
  catA,xgb,hgb w=(np.float64(0.74), np.float64(0.26), np.float64(0.0)) | AUC=0.89717 F1=0.200 (t=0.692)
  catA,xgb,hgb w=(np.float64(0.84), np.float64(0.16), np.float64(0.0)) | AUC=0.89717 F1=0.195 (t=0.692)
  catA,xgb,hgb w=(np.float64(0.88), np.float64(0.1), np.float64(0.02)) | AUC=0.89711 F1=0.205 (t=0.692)
  catA,xgb,hgb w=(np.float64(0.8), np.float64(0.14), np.float64(0.06)) | AUC=0.89711 F1=0.200 (t=0.692)
  catA,xgb,hgb w=(np.float64(0.76), np.float64(0.24), np.float64(0.0)) | AUC=0.89718 F1=0.200 (t=0.692)

=== 3-way (catB, xgb, hgb) simplex search (step 0.02) ===
Top 5 3-way blends (catB):
  catB,xgb,hgb w=(np.float64(0.86), np.float64(0.14), np.float64(0.0)) | AUC=0.89697 F1=0.205 (t=0.693)
  catB,xgb,hgb w=(np.float64(0.92), np.float64(0.08), np.float64(0.0)) | AUC=0.89687 F1=0.190 (t=0.693)
  catB,xgb,hgb w=(np.float64(0.9), np.float64(0.1), np.float64(0.0)) | AUC=0.89690 F1=0.190 (t=0.693)
  catB

## 14. Threshold Optimization (already folded into the ensemble search)


In [40]:
print('Single-model baselines (for reference):')
single_results = []
for name, oof_ in [('catA', oof_catA), ('catB', oof_catB), ('xgb', oof_xgb), ('hgb', oof_hgb), ('mlp', oof_mlp)]:
    t, f1, auc = report_oof(y, oof_, f'{name} (single)')
    single_results.append((name, auc, t, f1))

print('\n=== Combining all blend candidates ===')
all_candidates = []
all_candidates += blend_results      # catA-3w
all_candidates += blend_results_B    # catB-3w
all_candidates += blend_results_4    # catA-4w
all_candidates += blend_results_4B   # catB-4w
for name, auc, t, f1 in single_results:
    all_candidates.append((name, (1.0,), auc, t, f1))
all_candidates.sort(key=lambda r: r[4], reverse=True)
print('Top 10 overall candidates (sorted by OOF Macro F1):')
for r in all_candidates[:10]:
    print(f'  {r[0]:20s} w={r[1]} | AUC={r[2]:.5f} F1={r[3]:.3f} (t={r[4]:.3f})')

BEST_NAME, BEST_WEIGHTS, BEST_AUC, BEST_T, BEST_F1 = all_candidates[0]
print(f'\nBest candidate: {BEST_NAME} | weights={BEST_WEIGHTS} | AUC={BEST_AUC:.5f} | F1={BEST_F1:.5f} | t={BEST_T:.3f}')


Single-model baselines (for reference):
  catA (single)                          | AUC=0.89685 | MacroF1=0.69206 | t=0.205 | pos%=3.78
  catB (single)                          | AUC=0.89663 | MacroF1=0.69237 | t=0.185 | pos%=4.31
  xgb (single)                           | AUC=0.89280 | MacroF1=0.68711 | t=0.180 | pos%=4.80
  hgb (single)                           | AUC=0.88396 | MacroF1=0.67350 | t=0.175 | pos%=4.51
  mlp (single)                           | AUC=0.87461 | MacroF1=0.55439 | t=0.500 | pos%=23.63

=== Combining all blend candidates ===
Top 10 overall candidates (sorted by OOF Macro F1):
  catB,xgb,hgb,mlp     w=(np.float64(0.8500000000000001), np.float64(0.0), np.float64(0.0), np.float64(0.15000000000000002)) | AUC=0.89212 F1=0.310 (t=0.693)
  catB,xgb,hgb,mlp     w=(np.float64(0.8), np.float64(0.1), np.float64(0.0), np.float64(0.1)) | AUC=0.89393 F1=0.275 (t=0.693)
  catA,xgb,hgb,mlp     w=(np.float64(0.7000000000000001), np.float64(0.05), np.float64(0.0), np.float64(0.2

## 15. Optional Pseudo-Labeling Experiment


**Strictly OOF-validated.** We never touch the original test labels (they don't exist).
We only add **very high-confidence** test rows (P<0.01 → negative, P>0.99 → positive) to the training set in each fold,
**only using the OOF probability from the current best model** (which itself is OOF, so no test leakage).
Then we retrain CatBoost on the augmented training set and compare OOF Macro F1 on the **non-pseudo rows**.

If the new OOF Macro F1 is **worse** than the baseline, we discard pseudo-labeling.


In [41]:
use_pseudo = False
_is_catcentered = ('cat' in str(BEST_NAME).lower())
if _is_catcentered:
    # Build the best single CatBoost OOF to derive confident pseudo labels from test
    base_oof = oof_catA if 'catA' in BEST_NAME else oof_catB
    def weights_to_test(weights, test_lists):
        return sum(w * t for w, t in zip(weights, test_lists))
    if BEST_NAME == 'catA (single)':
        test_pred_for_pl = test_catA
    elif BEST_NAME == 'catB (single)':
        test_pred_for_pl = test_catB
    elif BEST_NAME == 'catA,xgb,hgb':
        test_pred_for_pl = weights_to_test(BEST_WEIGHTS, [test_catA, test_xgb, test_hgb])
    elif BEST_NAME == 'catB,xgb,hgb':
        test_pred_for_pl = weights_to_test(BEST_WEIGHTS, [test_catB, test_xgb, test_hgb])
    elif BEST_NAME == 'catA,xgb,hgb,mlp':
        test_pred_for_pl = weights_to_test(BEST_WEIGHTS, [test_catA, test_xgb, test_hgb, test_mlp])
    elif BEST_NAME == 'catB,xgb,hgb,mlp':
        test_pred_for_pl = weights_to_test(BEST_WEIGHTS, [test_catB, test_xgb, test_hgb, test_mlp])
    else:
        test_pred_for_pl = test_catA

    neg_mask = test_pred_for_pl < 0.01
    pos_mask = test_pred_for_pl > 0.99
    print(f'Pseudo-labels: {neg_mask.sum()} negatives, {pos_mask.sum()} positives from test.')

    # Pseudo set as extension of train (we pick the better of catA/catB feature view)
    base_view = X_catB_df if 'catB' in BEST_NAME else X_cat_df
    base_view_test = X_test_catB_df if 'catB' in BEST_NAME else X_test_cat_df
    pseudo_X = pd.concat([base_view_test.loc[neg_mask | pos_mask].reset_index(drop=True)], axis=0)
    pseudo_y = np.concatenate([np.zeros(neg_mask.sum(), dtype=np.int64),
                               np.ones(pos_mask.sum(), dtype=np.int64)])
    print(f'Pseudo training set: {pseudo_X.shape}  positives={int(pseudo_y.sum())}  negatives={int((pseudo_y==0).sum())}')

    def train_cat_pseudo(params, X_df, y_full, pseudo_X, pseudo_y, test_pred_for_pl, cat_idx):
        oof = np.zeros(len(X_df), dtype=np.float32)
        for fold, (tr, va) in enumerate(FOLDS):
            Xtr = pd.concat([X_df.iloc[tr], pseudo_X], axis=0).reset_index(drop=True)
            ytr = np.concatenate([y_full[tr], pseudo_y])
            Xva = X_df.iloc[va].reset_index(drop=True)
            m = CatBoostClassifier(loss_function='Logloss', eval_metric='AUC',
                                   iterations=4000, verbose=False, early_stopping_rounds=200, **params)
            m.fit(Xtr, ytr, eval_set=(Xva, y_full[va]), cat_features=cat_idx, use_best_model=True, verbose=False)
            pv = m.predict_proba(Xva)[:, 1]
            oof[va] = pv
        auc = roc_auc_score(y_full, oof)
        t, f1 = best_threshold(y_full, oof)
        return oof, auc, t, f1

    pl_oof, pl_auc, pl_t, pl_f1 = train_cat_pseudo(BEST_CAT_PARAMS, base_view, y, pseudo_X, pseudo_y, test_pred_for_pl, CAT_IDX)
    print(f'\nPseudo-labeled CatBoost OOF: AUC={pl_auc:.5f} F1={pl_f1:.5f} t={pl_t:.3f}')
    # Compare apples-to-apples: pseudo vs the *same* single CatBoost baseline used here.
    base_f1 = f1_catB if 'catB' in BEST_NAME else f1_catA
    base_auc = auc_catB if 'catB' in BEST_NAME else auc_catA
    base_t = t_catB if 'catB' in BEST_NAME else t_catA
    print(f'Baseline  CatBoost OOF : AUC={base_auc:.5f} F1={base_f1:.5f} t={base_t:.3f}')
    use_pseudo = (pl_f1 > base_f1) and (pl_f1 - base_f1) > 1e-4
    print(f'Use pseudo-labeling? {use_pseudo}')
else:
    print('Skipping pseudo-labeling (best is not a CatBoost-centered blend).')

if use_pseudo:
    # Recompute test predictions from the pseudo-labeled model.
    test_pred_pl = np.zeros(len(base_view_test), dtype=np.float32)
    for fold, (tr, va) in enumerate(FOLDS):
        Xtr = pd.concat([base_view.iloc[tr], pseudo_X], axis=0).reset_index(drop=True)
        ytr = np.concatenate([y[tr], pseudo_y])
        Xva = base_view.iloc[va].reset_index(drop=True)
        Xte = base_view_test.reset_index(drop=True)
        m = CatBoostClassifier(loss_function='Logloss', eval_metric='AUC',
                               iterations=4000, verbose=False, early_stopping_rounds=200, **BEST_CAT_PARAMS)
        m.fit(Xtr, ytr, eval_set=(Xva, y[va]), cat_features=CAT_IDX, use_best_model=True, verbose=False)
        test_pred_pl += m.predict_proba(Xte)[:, 1] / N_SPLITS
    # Substitute the cat-related test prediction with the pseudo-labeled one in the final blend.
    print(f'  Pseudo-labeled CatBoost test predictions computed. Shape: {test_pred_pl.shape}, mean={test_pred_pl.mean():.4f}')
    if 'catB' in BEST_NAME:
        test_catB = test_pred_pl  # replace in-place
    else:
        test_catA = test_pred_pl


Pseudo-labels: 14825 negatives, 0 positives from test.
Pseudo training set: (14825, 336)  positives=0  negatives=14825


Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU
Default metric period is 5 because AUC is/are not implemented for GPU



Pseudo-labeled CatBoost OOF: AUC=0.89551 F1=0.68849 t=0.205
Baseline  CatBoost OOF : AUC=0.89663 F1=0.69237 t=0.185
Use pseudo-labeling? False


## 16. Final Model Selection


In [42]:
print('='*60)
print('FINAL MODEL SUMMARY')
print('='*60)
print(f'Best candidate: {BEST_NAME}')
print(f'Weights:        {BEST_WEIGHTS}')
print(f'OOF AUC:        {BEST_AUC:.5f}')
print(f'OOF Macro F1:   {BEST_F1:.5f}')
print(f'Best threshold: {BEST_T:.4f}')
print(f'Use pseudo:     {use_pseudo}')

# Build the final OOF and final test prediction using the best weights
def pick_test_arrays(name, weights):
    if name == 'catA (single)':
        return [test_catA], weights
    if name == 'catB (single)':
        return [test_catB], weights
    if name == 'catA,xgb,hgb':
        return [test_catA, test_xgb, test_hgb], weights
    if name == 'catB,xgb,hgb':
        return [test_catB, test_xgb, test_hgb], weights
    if name == 'catA,xgb,hgb,mlp':
        return [test_catA, test_xgb, test_hgb, test_mlp], weights
    if name == 'catB,xgb,hgb,mlp':
        return [test_catB, test_xgb, test_hgb, test_mlp], weights
    raise ValueError(f'unknown name {name}')

def pick_oof_arrays(name, weights):
    if name == 'catA (single)':
        return [oof_catA], weights
    if name == 'catB (single)':
        return [oof_catB], weights
    if name == 'catA,xgb,hgb':
        return [oof_catA, oof_xgb, oof_hgb], weights
    if name == 'catB,xgb,hgb':
        return [oof_catB, oof_xgb, oof_hgb], weights
    if name == 'catA,xgb,hgb,mlp':
        return [oof_catA, oof_xgb, oof_hgb, oof_mlp], weights
    if name == 'catB,xgb,hgb,mlp':
        return [oof_catB, oof_xgb, oof_hgb, oof_mlp], weights
    raise ValueError(f'unknown name {name}')

test_lists, _ = pick_test_arrays(BEST_NAME, BEST_WEIGHTS)
oof_lists,  _ = pick_oof_arrays(BEST_NAME, BEST_WEIGHTS)
FINAL_OOF  = sum(w * o for w, o in zip(BEST_WEIGHTS, oof_lists))
FINAL_TEST = sum(w * t for w, t in zip(BEST_WEIGHTS, test_lists))
print(f'FINAL_OOF shape: {FINAL_OOF.shape}  | FINAL_TEST shape: {FINAL_TEST.shape}')

# Re-validate threshold on the final OOF (just to be safe)
FINAL_T, FINAL_F1, FINAL_AUC = report_oof(y, FINAL_OOF, 'FINAL OOF')
BEST_T = FINAL_T; BEST_F1 = FINAL_F1; BEST_AUC = FINAL_AUC
print(f'Confirmed final OOF: AUC={FINAL_AUC:.5f} F1={FINAL_F1:.5f} t={FINAL_T:.3f}')


FINAL MODEL SUMMARY
Best candidate: catB,xgb,hgb,mlp
Weights:        (np.float64(0.8500000000000001), np.float64(0.0), np.float64(0.0), np.float64(0.15000000000000002))
OOF AUC:        0.89212
OOF Macro F1:   0.69283
Best threshold: 0.3100
Use pseudo:     False
FINAL_OOF shape: (76020,)  | FINAL_TEST shape: (60654,)
  FINAL OOF                              | AUC=0.89212 | MacroF1=0.69283 | t=0.310 | pos%=3.49
Confirmed final OOF: AUC=0.89212 F1=0.69283 t=0.310


## 17. Test Prediction


In [43]:
final_pred = (FINAL_TEST >= BEST_T).astype(int)
print(f'Final threshold: {BEST_T:.3f}')
print(f'Predicted class distribution: {np.bincount(final_pred)}')
pos_pct = final_pred.mean() * 100.0
print(f'Predicted positive percentage: {pos_pct:.2f}%')


Final threshold: 0.310
Predicted class distribution: [58979  1675]
Predicted positive percentage: 2.76%


## 18. Submission Generation


In [46]:
sub = pd.DataFrame({'id': test_ids, 'TARGET': final_pred})
sub = sample_sub[['id']].merge(sub, on='id', how='left')
assert sub['TARGET'].notna().all(), 'Missing predictions for some test ids!'
sub['TARGET'] = sub['TARGET'].astype(int)
print(f'Submission shape: {sub.shape}  | cols: {sub.columns.tolist()}')
print(f'Predicted positive %: {sub["TARGET"].mean()*100:.2f}%')
print(sub.head())

SUB_PATH = os.path.join(OUT_DIR, 'submission2.csv')
sub.to_csv(SUB_PATH, index=False)
print(f'Saved submission to: {SUB_PATH}')

# Save artefacts
np.savez(os.path.join(OUT_DIR, 'oof_predictions.npz'),
         oof_catA=oof_catA, oof_catB=oof_catB, oof_xgb=oof_xgb, oof_hgb=oof_hgb, oof_mlp=oof_mlp,
         final_oof=FINAL_OOF)
np.savez(os.path.join(OUT_DIR, 'test_predictions.npz'),
         test_catA=test_catA, test_catB=test_catB, test_xgb=test_xgb, test_hgb=test_hgb, test_mlp=test_mlp,
         final_test=FINAL_TEST)
with open(os.path.join(OUT_DIR, 'best_weights.json'), 'w') as f:
    json.dump({'name': BEST_NAME, 'weights': list(map(float, BEST_WEIGHTS))}, f, indent=2)
with open(os.path.join(OUT_DIR, 'best_threshold.json'), 'w') as f:
    json.dump({'threshold': float(BEST_T), 'macro_f1': float(BEST_F1), 'auc': float(BEST_AUC)}, f, indent=2)
print('Saved oof_predictions.npz, test_predictions.npz, best_weights.json, best_threshold.json')


Submission shape: (60654, 2)  | cols: ['id', 'TARGET']
Predicted positive %: 2.76%
      id  TARGET
0   3496       0
1  17271       0
2  44259       0
3  64996       0
4  23333       0
Saved submission to: d:\DS\kaggle\PSTU_Datathon\submission2.csv
Saved oof_predictions.npz, test_predictions.npz, best_weights.json, best_threshold.json


## 19. Final Results Table


In [45]:
def fmt_row(name, auc, t, f1, test_pred):
    pos = (test_pred >= t).mean() * 100.0 if test_pred is not None else float('nan')
    return f'{name:38s} | AUC={auc:.5f} | F1={f1:.5f} | t={t:.3f} | pos%={pos:.2f}'

print('Model/Ensemble                            | AUC     | MacroF1 | thr  | pos%')
print('-'*92)
rows = []
rows.append(('CatBoost (original, 5-seed)', auc_catA, t_catA, f1_catA, test_catA))
rows.append(('CatBoost (+row stats, 5-seed)', auc_catB, t_catB, f1_catB, test_catB))
rows.append(('XGBoost (5-seed, +row stats)', auc_xgb, t_xgb, f1_xgb, test_xgb))
rows.append(('HGB (+row stats)', auc_hgb, t_hgb, f1_hgb, test_hgb))
rows.append(('MLP', auc_mlp, t_mlp, f1_mlp, test_mlp))
for r in blend_results[:3]:
    name, w, auc, t, f1 = r
    test_pred = sum(wi * tp for wi, tp in zip(w, [test_catA, test_xgb, test_hgb]))
    rows.append((f'{name} {tuple(round(float(x),2) for x in w)}', auc, t, f1, test_pred))
for r in blend_results_B[:3]:
    name, w, auc, t, f1 = r
    test_pred = sum(wi * tp for wi, tp in zip(w, [test_catB, test_xgb, test_hgb]))
    rows.append((f'{name} {tuple(round(float(x),2) for x in w)}', auc, t, f1, test_pred))
for r in blend_results_4[:3]:
    name, w, auc, t, f1 = r
    test_pred = sum(wi * tp for wi, tp in zip(w, [test_catA, test_xgb, test_hgb, test_mlp]))
    rows.append((f'{name} {tuple(round(float(x),2) for x in w)}', auc, t, f1, test_pred))
for r in blend_results_4B[:3]:
    name, w, auc, t, f1 = r
    test_pred = sum(wi * tp for wi, tp in zip(w, [test_catB, test_xgb, test_hgb, test_mlp]))
    rows.append((f'{name} {tuple(round(float(x),2) for x in w)}', auc, t, f1, test_pred))
for r in rows:
    print(fmt_row(*r))

print()
print('='*60)
print('FINAL MODEL')
print('='*60)
w = list(BEST_WEIGHTS) if not isinstance(BEST_WEIGHTS, (int, float)) else [float(BEST_WEIGHTS)]
w_catB = w_xgb = w_hgb = w_mlp = 0.0
if BEST_NAME == 'catA (single)': w_catA = w[0]
elif BEST_NAME == 'catB (single)': w_catB = w[0]
elif BEST_NAME == 'catA,xgb,hgb':
    w_catA, w_xgb, w_hgb = w
elif BEST_NAME == 'catB,xgb,hgb':
    w_catB, w_xgb, w_hgb = w
elif BEST_NAME == 'catA,xgb,hgb,mlp':
    w_catA, w_xgb, w_hgb, w_mlp = w
elif BEST_NAME == 'catB,xgb,hgb,mlp':
    w_catB, w_xgb, w_hgb, w_mlp = w
print(f'Model:                {BEST_NAME}')
print(f'OOF AUC:              {BEST_AUC:.5f}')
print(f'OOF Macro F1:         {BEST_F1:.5f}')
print(f'Optimal threshold:    {BEST_T:.4f}')
print(f'CatBoost weight:      {w_catA:.3f}')
print(f'CatBoost-B weight:    {w_catB:.3f}')
print(f'XGBoost weight:       {w_xgb:.3f}')
print(f'HGB weight:           {w_hgb:.3f}')
print(f'MLP weight:           {w_mlp:.3f}')
print(f'Predicted positive %: {pos_pct:.2f}')
print(f'Submission path:      {SUB_PATH}')
print('='*60)


Model/Ensemble                            | AUC     | MacroF1 | thr  | pos%
--------------------------------------------------------------------------------------------
CatBoost (original, 5-seed)            | AUC=0.89685 | F1=0.69206 | t=0.205 | pos%=3.09
CatBoost (+row stats, 5-seed)          | AUC=0.89663 | F1=0.69237 | t=0.185 | pos%=3.55
XGBoost (5-seed, +row stats)           | AUC=0.89280 | F1=0.68711 | t=0.180 | pos%=4.14
HGB (+row stats)                       | AUC=0.88396 | F1=0.67350 | t=0.175 | pos%=3.77
MLP                                    | AUC=0.87461 | F1=0.55439 | t=0.500 | pos%=22.56
catA,xgb,hgb (0.74, 0.26, 0.0)         | AUC=0.89717 | F1=0.69249 | t=0.200 | pos%=3.28
catA,xgb,hgb (0.84, 0.16, 0.0)         | AUC=0.89717 | F1=0.69245 | t=0.195 | pos%=3.38
catA,xgb,hgb (0.88, 0.1, 0.02)         | AUC=0.89711 | F1=0.69244 | t=0.205 | pos%=3.11
catB,xgb,hgb (0.86, 0.14, 0.0)         | AUC=0.89697 | F1=0.69265 | t=0.205 | pos%=3.12
catB,xgb,hgb (0.92, 0.08, 0.0)        

NameError: name 'w_catA' is not defined